# Risk filter checks - 1h candles

        Hypothesis: **microstructure is useful as a risk filter if momentum top-K candidates perform better in low-risk microstructure states than in high-risk states.**

        Momentum score: `vol_adj_momentum_24`. Risk features: `vpin`, `roll_impact`, `amihud`, `roll_measure`, and a combined microstructure risk rank. Horizons are 6h, 12h, and 24h forward returns measured on 1h candles.

        This notebook uses cached 1h Binance spot candles and cached feature calculations. It is research-only; live trading should compute the same features on fresh in-memory candles instead of reading CSVs.


## What Would Confirm The Hypothesis?

        We should observe:

        - top-K momentum candidates in **low** microstructure-risk terciles have higher mean or median forward returns than **high** risk terciles;
        - momentum IC inside low-risk states is better than inside high-risk states;
        - a simple gated or scaled policy improves net top-K returns after a cost haircut;
        - the improvement is not only one calendar slice or one risk variable.


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import HTML, display

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = next(parent for parent in (NOTEBOOK_DIR, *NOTEBOOK_DIR.parents) if (parent / "bot").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from bot.forward_ic import add_forward_return, information_coefficient  # noqa: E402

FEATURE_PATH = REPO_ROOT / "notebooks/feature/feature_checks_1h_h6_features.csv"
OUTPUT_DIR = REPO_ROOT / "notebooks/microstructure"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HORIZONS = (6, 12, 24)
TOP_KS = (3, 5)
MOMENTUM_FEATURE = "vol_adj_momentum_24"
RISK_FEATURES = ("vpin", "roll_impact", "amihud", "roll_measure")
COST_BPS = 20
MIN_ASSETS_PER_TIMESTAMP = 8


In [ ]:
def load_base() -> pd.DataFrame:
    df = pd.read_csv(FEATURE_PATH)
    df = df.sort_values(["pair", "open_time"]).reset_index(drop=True)
    for horizon in HORIZONS:
        target = f"forward_return_{horizon}"
        if target not in df.columns:
            df = add_forward_return(
                df,
                horizon=horizon,
                price_col="close",
                group_col="pair",
                sort_col="open_time",
                target_col=target,
            )

    rank_cols = []
    for col in RISK_FEATURES:
        rank_col = f"{col}_risk_rank"
        df[rank_col] = df.groupby("open_time", observed=True)[col].rank(pct=True)
        rank_cols.append(rank_col)

    df["combined_micro_risk"] = df[rank_cols].mean(axis=1)
    df["combined_micro_risk_rank"] = df.groupby("open_time", observed=True)[
        "combined_micro_risk"
    ].rank(pct=True)
    for col in (*RISK_FEATURES, "combined_micro_risk"):
        df[f"{col}_risk_tercile"] = pd.cut(
            df[risk_rank_col(col)],
            bins=[0.0, 1 / 3, 2 / 3, 1.0],
            labels=["low", "mid", "high"],
            include_lowest=True,
        )
    return df


def risk_rank_col(risk_col: str) -> str:
    if risk_col == "combined_micro_risk":
        return "combined_micro_risk_rank"
    return f"{risk_col}_risk_rank"


def summarize_returns(df: pd.DataFrame, target_col: str, group_cols: list[str]) -> pd.DataFrame:
    grouped = df.groupby(group_cols, observed=True)[target_col]
    return grouped.agg(
        observations="count",
        mean_return="mean",
        median_return="median",
        hit_rate=lambda s: float((s > 0).mean()),
        q05=lambda s: s.quantile(0.05),
        q95=lambda s: s.quantile(0.95),
    ).reset_index()


def top_k_by_score(df: pd.DataFrame, score_col: str, top_k: int) -> pd.DataFrame:
    ranked = df.dropna(subset=[score_col]).copy()
    ranked["score_rank"] = ranked.groupby("open_time", observed=True)[score_col].rank(
        ascending=False,
        method="first",
    )
    return ranked[ranked["score_rank"] <= top_k].copy()


def conditional_topk_returns(base: pd.DataFrame) -> pd.DataFrame:
    tables = []
    risk_names = (*RISK_FEATURES, "combined_micro_risk")
    for horizon in HORIZONS:
        target_col = f"forward_return_{horizon}"
        cols = [
            "open_time",
            "pair",
            MOMENTUM_FEATURE,
            target_col,
            *(f"{col}_risk_tercile" for col in risk_names),
        ]
        frame = base[cols].dropna(subset=[MOMENTUM_FEATURE, target_col]).copy()
        enough_assets = frame.groupby("open_time", observed=True)["pair"].transform("count")
        frame = frame[enough_assets >= MIN_ASSETS_PER_TIMESTAMP]
        for top_k in TOP_KS:
            selected = top_k_by_score(frame, MOMENTUM_FEATURE, top_k)
            for risk_col in risk_names:
                tercile_col = f"{risk_col}_risk_tercile"
                summary = summarize_returns(selected, target_col, [tercile_col])
                summary = summary.rename(columns={tercile_col: "risk_tercile"})
                summary["risk_feature"] = risk_col
                summary["horizon"] = horizon
                summary["top_k"] = top_k
                tables.append(summary)
    return pd.concat(tables, ignore_index=True)


def policy_candidates(
    frame: pd.DataFrame,
    risk_col: str,
    target_col: str,
    top_k: int,
    policy: str,
) -> pd.DataFrame:
    rank_col = risk_rank_col(risk_col)
    if policy == "pure_momentum":
        candidates = frame.copy()
        score_col = MOMENTUM_FEATURE
    elif policy == "gate_high_risk":
        candidates = frame[frame[rank_col] <= 2 / 3].copy()
        score_col = MOMENTUM_FEATURE
    elif policy == "low_risk_only":
        candidates = frame[frame[rank_col] <= 1 / 3].copy()
        score_col = MOMENTUM_FEATURE
    elif policy == "scaled_by_low_risk":
        candidates = frame.copy()
        score_col = "risk_scaled_momentum"
        candidates[score_col] = candidates[MOMENTUM_FEATURE] * (1 - candidates[rank_col])
    else:
        raise ValueError(f"Unknown policy: {policy}")
    selected = top_k_by_score(candidates, score_col, top_k)
    selected = selected.dropna(subset=[target_col])
    selected["policy"] = policy
    selected["risk_feature"] = risk_col
    return selected


def summarize_policy_returns(selected: pd.DataFrame, target_col: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    period_returns = selected.groupby(
        ["open_time", "policy", "risk_feature"],
        observed=True,
    )[target_col].mean()
    period_returns = period_returns.reset_index(name="portfolio_return")
    period_returns["net_return"] = period_returns["portfolio_return"] - COST_BPS / 10_000
    grouped = period_returns.groupby(["policy", "risk_feature"], observed=True)
    summary = grouped.agg(
        periods=("portfolio_return", "count"),
        mean_return=("portfolio_return", "mean"),
        median_return=("portfolio_return", "median"),
        std_return=("portfolio_return", "std"),
        hit_rate=("portfolio_return", lambda s: float((s > 0).mean())),
        mean_net_return=("net_return", "mean"),
        net_hit_rate=("net_return", lambda s: float((s > 0).mean())),
        q05=("portfolio_return", lambda s: s.quantile(0.05)),
        q95=("portfolio_return", lambda s: s.quantile(0.95)),
    ).reset_index()
    return summary, period_returns


def policy_comparison(base: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    summaries = []
    returns = []
    risk_names = (*RISK_FEATURES, "combined_micro_risk")
    policies = ("pure_momentum", "gate_high_risk", "low_risk_only", "scaled_by_low_risk")
    for horizon in HORIZONS:
        target_col = f"forward_return_{horizon}"
        needed = [
            "open_time",
            "pair",
            MOMENTUM_FEATURE,
            target_col,
            *(risk_rank_col(risk) for risk in risk_names),
        ]
        frame = base[needed].dropna(subset=[MOMENTUM_FEATURE, target_col]).copy()
        enough_assets = frame.groupby("open_time", observed=True)["pair"].transform("count")
        frame = frame[enough_assets >= MIN_ASSETS_PER_TIMESTAMP]
        for top_k in TOP_KS:
            selected_parts = [
                policy_candidates(frame, risk, target_col, top_k, policy)
                for risk in risk_names
                for policy in policies
            ]
            selected = pd.concat(selected_parts, ignore_index=True)
            summary, period_returns = summarize_policy_returns(selected, target_col)
            summary["horizon"] = horizon
            summary["top_k"] = top_k
            period_returns["horizon"] = horizon
            period_returns["top_k"] = top_k
            summaries.append(summary)
            returns.append(period_returns)
    return pd.concat(summaries, ignore_index=True), pd.concat(returns, ignore_index=True)


def cross_sectional_ic_by_time(
    df: pd.DataFrame,
    feature_col: str,
    target_col: str,
) -> pd.DataFrame:
    rows = []
    for open_time, group in df.groupby("open_time", observed=True):
        if group[feature_col].nunique(dropna=True) < 2 or group[target_col].nunique(dropna=True) < 2:
            continue
        rows.append(
            {
                "open_time": open_time,
                "pearson": information_coefficient(
                    group[feature_col],
                    group[target_col],
                    method="pearson",
                )[0],
                "spearman": information_coefficient(
                    group[feature_col],
                    group[target_col],
                    method="spearman",
                )[0],
                "assets": len(group),
            }
        )
    return pd.DataFrame(rows)


def conditional_momentum_ic(base: pd.DataFrame) -> pd.DataFrame:
    rows = []
    risk_names = (*RISK_FEATURES, "combined_micro_risk")
    for horizon in HORIZONS:
        target_col = f"forward_return_{horizon}"
        for risk in risk_names:
            tercile_col = f"{risk}_risk_tercile"
            for tercile, group in base.groupby(tercile_col, observed=True):
                frame = group.dropna(subset=[MOMENTUM_FEATURE, target_col])
                enough_assets = frame.groupby("open_time", observed=True)["pair"].transform("count")
                frame = frame[enough_assets >= 4]
                ic = cross_sectional_ic_by_time(frame, MOMENTUM_FEATURE, target_col)
                if ic.empty:
                    continue
                rows.append(
                    {
                        "horizon": horizon,
                        "risk_feature": risk,
                        "risk_tercile": str(tercile),
                        "periods": len(ic),
                        "mean_spearman": ic["spearman"].mean(),
                        "median_spearman": ic["spearman"].median(),
                        "spearman_hit_rate": (ic["spearman"] > 0).mean(),
                        "mean_pearson": ic["pearson"].mean(),
                        "median_pearson": ic["pearson"].median(),
                        "pearson_hit_rate": (ic["pearson"] > 0).mean(),
                    }
                )
    return pd.DataFrame(rows)


def monthly_policy_summary(policy_returns: pd.DataFrame) -> pd.DataFrame:
    frame = policy_returns.copy()
    frame["month"] = pd.to_datetime(frame["open_time"]).dt.to_period("M").astype(str)
    grouped = frame.groupby(
        ["horizon", "top_k", "risk_feature", "policy", "month"],
        observed=True,
    )
    return grouped.agg(
        periods=("portfolio_return", "count"),
        mean_return=("portfolio_return", "mean"),
        mean_net_return=("net_return", "mean"),
        hit_rate=("portfolio_return", lambda s: float((s > 0).mean())),
    ).reset_index()


In [1]:
base = load_base()
conditional_topk = conditional_topk_returns(base)
policy_summary, policy_returns = policy_comparison(base)
conditional_ic = conditional_momentum_ic(base)
monthly_summary = monthly_policy_summary(policy_returns)

conditional_topk.to_csv(OUTPUT_DIR / "risk_filter_checks_1h_conditional_topk.csv", index=False)
policy_summary.to_csv(OUTPUT_DIR / "risk_filter_checks_1h_policy_summary.csv", index=False)
policy_returns.to_csv(OUTPUT_DIR / "risk_filter_checks_1h_policy_returns.csv", index=False)
conditional_ic.to_csv(OUTPUT_DIR / "risk_filter_checks_1h_conditional_momentum_ic.csv", index=False)
monthly_summary.to_csv(OUTPUT_DIR / "risk_filter_checks_1h_monthly_summary.csv", index=False)

metadata = {
    "source_features": str(FEATURE_PATH),
    "candles": "1h Binance spot candles from cached research data",
    "momentum_feature": MOMENTUM_FEATURE,
    "risk_features": [*RISK_FEATURES, "combined_micro_risk"],
    "horizons": list(HORIZONS),
    "top_ks": list(TOP_KS),
    "cost_bps": COST_BPS,
    "min_assets_per_timestamp": MIN_ASSETS_PER_TIMESTAMP,
    "rows": int(len(base)),
    "pairs": int(base["pair"].nunique()),
    "start_open_time": str(base["open_time"].min()),
    "end_open_time": str(base["open_time"].max()),
}
(OUTPUT_DIR / "risk_filter_checks_1h_metadata.json").write_text(json.dumps(metadata, indent=2, sort_keys=True))
metadata


{
  "candles": "1h Binance spot candles from cached research data",
  "cost_bps": 20,
  "end_open_time": "1780689600000",
  "horizons": [
    6,
    12,
    24
  ],
  "min_assets_per_timestamp": 8,
  "momentum_feature": "vol_adj_momentum_24",
  "pairs": 43,
  "risk_features": [
    "vpin",
    "roll_impact",
    "amihud",
    "roll_measure",
    "combined_micro_risk"
  ],
  "rows": 43000,
  "source_features": "notebooks/feature/feature_checks_1h_h6_features.csv",
  "start_open_time": "1777093200000",
  "top_ks": [
    3,
    5
  ]
}


In [ ]:
def svg_grouped_bars(
    df: pd.DataFrame,
    title: str,
    label_col: str,
    value_col: str,
    width: int = 1080,
    row_h: int = 25,
    left: int = 350,
    right: int = 90,
    limit: int | None = 30,
) -> str:
    rows = df[[label_col, value_col]].dropna().copy().sort_values(value_col)
    if limit and len(rows) > limit:
        rows = pd.concat([rows.head(limit // 2), rows.tail(limit - limit // 2)]).sort_values(value_col)
    height = 50 + row_h * len(rows)
    plot_w = width - left - right
    max_abs = max(float(rows[value_col].abs().max()), 0.001)
    zero_x = left + plot_w / 2
    scale = (plot_w / 2) / max_abs
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" '
        f'viewBox="0 0 {width} {height}">'
    ]
    parts.append('<rect width="100%" height="100%" fill="#fff"/>')
    parts.append(
        f'<text x="12" y="24" font-family="Arial" font-size="16" '
        f'font-weight="700">{title}</text>'
    )
    parts.append(f'<line x1="{zero_x:.1f}" y1="38" x2="{zero_x:.1f}" y2="{height - 12}" stroke="#555"/>')
    for i, row in enumerate(rows.itertuples(index=False)):
        label = str(getattr(row, label_col)).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
        val = float(getattr(row, value_col))
        y = 46 + i * row_h
        bw = abs(val) * scale
        x = zero_x if val >= 0 else zero_x - bw
        color = "#2f7d32" if val >= 0 else "#b23a3a"
        parts.append(f'<text x="12" y="{y + 13}" font-family="Arial" font-size="11">{label}</text>')
        parts.append(f'<rect x="{x:.1f}" y="{y}" width="{bw:.1f}" height="16" fill="{color}" opacity="0.82"/>')
        tx = zero_x + bw + 6 if val >= 0 else zero_x - bw - 60
        parts.append(f'<text x="{tx:.1f}" y="{y + 13}" font-family="Arial" font-size="11">{val:+.4f}</text>')
    parts.append("</svg>")
    return "".join(parts)


## Conditional Top-K Returns

        This first selects top-K candidates by `vol_adj_momentum_24`, then buckets those selected names by microstructure risk. This is the cleanest test of whether microstructure tells us when to trust the momentum signal.

        Read this as: among the names momentum wanted to buy, did low-risk names perform better than high-risk names?


In [1]:
conditional_topk[
    (conditional_topk["top_k"] == 3)
    & (conditional_topk["risk_feature"].isin(["vpin", "roll_impact", "combined_micro_risk"]))
].sort_values(["horizon", "risk_feature", "risk_tercile"])


risk_tercile  observations  mean_return  median_return  hit_rate       q05      q95        risk_feature  horizon  top_k
         low           641     0.003051       0.000564  0.525741 -0.038620 0.053820 combined_micro_risk        6      3
         mid           934     0.001810      -0.001129  0.460385 -0.053594 0.067165 combined_micro_risk        6      3
        high          1257     0.001641      -0.001053  0.467780 -0.049420 0.061216 combined_micro_risk        6      3
         low          1110     0.002243       0.000000  0.487387 -0.052926 0.064841         roll_impact        6      3
         mid           786     0.002091      -0.001044  0.469466 -0.059054 0.078057         roll_impact        6      3
        high           933     0.001687      -0.000574  0.474812 -0.027003 0.040451         roll_impact        6      3
         low           489     0.003318       0.000286  0.525562 -0.030509 0.048962                vpin        6      3
         mid           513     0.002177 

In [1]:
conditional_focus = conditional_topk[
    (conditional_topk["top_k"] == 3)
    & (conditional_topk["risk_feature"].isin(["vpin", "roll_impact", "combined_micro_risk"]))
].copy()
conditional_focus["label"] = conditional_focus.apply(
    lambda r: f"h{int(r['horizon'])} {r['risk_feature']} {r['risk_tercile']}",
    axis=1,
)
display(HTML(svg_grouped_bars(conditional_focus, "Top-3 momentum candidate return by risk tercile", "label", "mean_return")))


Top-3 momentum candidate return by risk tercile h6 combined_micro_risk high +0.0016 h6 roll_impact high +0.0017 h6 combined_micro_risk mid +0.0018 h6 vpin high +0.0018 h6 roll_impact mid +0.0021 h6 vpin mid +0.0022 h6 roll_impact low +0.0022 h6 combined_micro_risk low +0.0031 h12 roll_impact high +0.0031 h6 vpin low +0.0033 h12 combined_micro_risk mid +0.0046 h12 combined_micro_risk high +0.0047 h24 roll_impact high +0.0048 h12 vpin high +0.0054 h12 roll_impact low +0.0060 h12 vpin low +0.0063 h12 vpin mid +0.0072 h24 combined_micro_risk high +0.0072 h12 roll_impact mid +0.0073 h12 combined_micro_risk low +0.0080 h24 vpin high +0.0103 h24 roll_impact low +0.0117 h24 vpin mid +0.0125 h24 combined_micro_risk mid +0.0129 h24 combined_micro_risk low +0.0188 h24 roll_impact mid +0.0199 h24 vpin low +0.0209

## Policy Comparison

        Policies tested:

        - `pure_momentum`: choose top-K by momentum only.
        - `gate_high_risk`: drop high-risk names, then choose top-K by momentum.
        - `low_risk_only`: only choose from the low-risk tercile.
        - `scaled_by_low_risk`: score = momentum * (1 - risk_rank).

        `mean_net_return` subtracts a simple 20 bps holding-period cost haircut. This is not a full execution model, but it keeps the comparison from rewarding fragile gross returns.


In [1]:
policy_summary[
    (policy_summary["top_k"] == 3)
    & (policy_summary["risk_feature"].isin(["vpin", "roll_impact", "combined_micro_risk"]))
].sort_values(["horizon", "risk_feature", "policy"])


            policy        risk_feature  periods  mean_return  median_return  std_return  hit_rate  mean_net_return  net_hit_rate       q05      q95  horizon  top_k
    gate_high_risk combined_micro_risk      944     0.001830       0.001071    0.023347  0.525424        -0.000170      0.471398 -0.032134 0.042871        6      3
     low_risk_only combined_micro_risk      944     0.000800       0.000485    0.018811  0.515890        -0.001200      0.444915 -0.027779 0.027230        6      3
     pure_momentum combined_micro_risk      970     0.001900       0.000453    0.024268  0.515464        -0.000100      0.457732 -0.034987 0.045985        6      3
scaled_by_low_risk combined_micro_risk      944     0.001456      -0.000009    0.022907  0.500000        -0.000544      0.438559 -0.032987 0.041652        6      3
    gate_high_risk         roll_impact      943     0.002846       0.001416    0.027006  0.534464         0.000846      0.485684 -0.037412 0.054473        6      3
     low_risk_on

In [1]:
policy_focus = policy_summary[
    (policy_summary["top_k"] == 3)
    & (policy_summary["risk_feature"].isin(["vpin", "roll_impact", "combined_micro_risk"]))
].copy()
policy_focus["label"] = policy_focus.apply(
    lambda r: f"h{int(r['horizon'])} {r['risk_feature']} {r['policy']}",
    axis=1,
)
display(
    HTML(
        svg_grouped_bars(
            policy_focus,
            f"Policy mean net return after {COST_BPS} bps, top-3",
            "label",
            "mean_net_return",
            width=1120,
            left=430,
        )
    )
)


Policy mean net return after 20 bps, top-3 h12 vpin low_risk_only -0.0026 h6 vpin low_risk_only -0.0024 h6 combined_micro_risk low_risk_only -0.0012 h6 vpin gate_high_risk -0.0011 h6 roll_impact low_risk_only -0.0007 h6 combined_micro_risk scaled_by_low_risk -0.0005 h24 vpin low_risk_only -0.0004 h6 roll_impact scaled_by_low_risk -0.0004 h6 vpin scaled_by_low_risk -0.0003 h6 combined_micro_risk gate_high_risk -0.0002 h6 vpin pure_momentum -0.0001 h6 combined_micro_risk pure_momentum -0.0001 h6 roll_impact pure_momentum -0.0001 h12 vpin gate_high_risk -0.0001 h12 combined_micro_risk low_risk_only +0.0002 h12 combined_micro_risk pure_momentum +0.0030 h12 vpin pure_momentum +0.0030 h12 roll_impact pure_momentum +0.0030 h24 combined_micro_risk low_risk_only +0.0045 h12 roll_impact gate_high_risk +0.0053 h24 vpin gate_high_risk +0.0056 h24 roll_impact low_risk_only +0.0065 h24 vpin scaled_by_low_risk +0.0066 h24 combined_micro_risk scaled_by_low_risk +0.0083 h24 roll_impact pure_momentum +0.0089 h24 combined_micro_risk pure_momentum +0.0089 h24 vpin pure_momentum +0.0089 h24 combined_micro_risk gate_high_risk +0.0092 h24 roll_impact scaled_by_low_risk +0.0099 h24 roll_impact gate_high_risk +0.0136

## Conditional Momentum IC

        This checks whether the ranking quality of `vol_adj_momentum_24` changes inside each microstructure-risk tercile. If microstructure is a good risk filter, low-risk terciles should usually show stronger positive IC than high-risk terciles.


In [1]:
conditional_ic[
    conditional_ic["risk_feature"].isin(["vpin", "roll_impact", "combined_micro_risk"])
].sort_values(["horizon", "risk_feature", "risk_tercile"])


 horizon        risk_feature risk_tercile  periods  mean_spearman  median_spearman  spearman_hit_rate  mean_pearson  median_pearson  pearson_hit_rate
       6 combined_micro_risk         high      944       0.031943         0.035714           0.534958      0.038230        0.041107          0.534958
       6 combined_micro_risk          low      944       0.069172         0.072527           0.592161      0.084772        0.081815          0.572034
       6 combined_micro_risk          mid      944       0.047071         0.054945           0.562500      0.054273        0.066653          0.565678
       6         roll_impact         high      943       0.040898         0.028571           0.532344      0.059462        0.057158          0.565217
       6         roll_impact          low      943       0.062052         0.076923           0.584305      0.078763        0.096055          0.572641
       6         roll_impact          mid      943       0.055966         0.072527           0.58642

In [1]:
ic_focus = conditional_ic[
    conditional_ic["risk_feature"].isin(["vpin", "roll_impact", "combined_micro_risk"])
].copy()
ic_focus["label"] = ic_focus.apply(
    lambda r: f"h{int(r['horizon'])} {r['risk_feature']} {r['risk_tercile']}",
    axis=1,
)
display(HTML(svg_grouped_bars(ic_focus, "Momentum Spearman IC inside risk terciles", "label", "mean_spearman")))


Momentum Spearman IC inside risk terciles h6 combined_micro_risk high +0.0319 h12 combined_micro_risk high +0.0404 h6 roll_impact high +0.0409 h6 vpin high +0.0429 h6 vpin low +0.0431 h24 roll_impact high +0.0451 h6 combined_micro_risk mid +0.0471 h24 combined_micro_risk high +0.0472 h6 vpin mid +0.0536 h6 roll_impact mid +0.0560 h12 roll_impact high +0.0579 h6 roll_impact low +0.0621 h12 vpin low +0.0632 h6 combined_micro_risk low +0.0692 h24 vpin low +0.0766 h12 combined_micro_risk mid +0.0785 h12 vpin high +0.0801 h12 vpin mid +0.0867 h12 roll_impact mid +0.0934 h24 combined_micro_risk mid +0.1003 h12 roll_impact low +0.1041 h24 vpin high +0.1061 h24 vpin mid +0.1078 h12 combined_micro_risk low +0.1147 h24 roll_impact mid +0.1379 h24 roll_impact low +0.1661 h24 combined_micro_risk low +0.1835

## Monthly Stability Slice

        This is a minimal stability check. With roughly 1000 hourly bars, it is not enough for a final conclusion, but it quickly shows whether a policy result is dominated by one calendar slice.


In [1]:
monthly_summary[
    (monthly_summary["top_k"] == 3) & (monthly_summary["risk_feature"].eq("vpin"))
].sort_values(["horizon", "policy", "month"])


 horizon  top_k risk_feature             policy   month  periods  mean_return  mean_net_return  hit_rate
       6      3         vpin     gate_high_risk 1970-01      895     0.000912        -0.001088  0.516201
       6      3         vpin      low_risk_only 1970-01      895    -0.000448        -0.002448  0.518436
       6      3         vpin      pure_momentum 1970-01      970     0.001900        -0.000100  0.515464
       6      3         vpin scaled_by_low_risk 1970-01      895     0.001666        -0.000334  0.519553
      12      3         vpin     gate_high_risk 1970-01      889     0.001925        -0.000075  0.533183
      12      3         vpin      low_risk_only 1970-01      889    -0.000626        -0.002626  0.515186
      12      3         vpin      pure_momentum 1970-01      964     0.005031         0.003031  0.539419
      12      3         vpin scaled_by_low_risk 1970-01      889     0.003688         0.001688  0.525309
      24      3         vpin     gate_high_risk 1970-01

## First-Pass Read

        The hypothesis is only partially supported. Conditional top-K tables are the most direct evidence: some high-risk buckets are worse than low or mid buckets, especially at longer horizons, but the ordering is not reliably monotonic across risk variables.

        The policy table is stricter. If `gate_high_risk` or `scaled_by_low_risk` does not improve `mean_net_return` versus `pure_momentum`, then microstructure is not yet strong enough as a hard entry filter. In that case, the better next test is position sizing or exposure throttling rather than binary filtering.
